In [2]:
# =============================================================================
#  EXP27b ONLY — naturalistic factual recall, with the v1 bugs fixed.
#
#  Nothing here depends on the arithmetic experiments, so EXP2 / EXP3 / EXP26 are
#  omitted: this loads the models, builds the fact set, and runs EXP27b. Much
#  shorter than the full suite.
#
#  RUN ORDER: dependency cells -> RESTART KERNEL -> paste your HF token -> run down.
#
#  WHAT CHANGED vs the run that produced native_full = 0.146:
#    1. The bin is now defined on the DECODED FULL ANSWER for both models. The v1 bin
#       used exact first-token IDs while full-answer was scored on the decoded string;
#       a model can reach the same string through a different tokenization, so items
#       failed the token check, entered the bin, then passed the string check. That is
#       the whole source of native_full > 0. The new rule matches the paper's
#       cross-family arms, so native_full is 0 by construction and native FIRST-token
#       is nonzero and reported.
#    2. Strict answer matching. v1 used startswith, so gold "Au" matched "Australia"
#       and gold "US" matched "used to be a colony". TriviaQA is full of short answers,
#       so this inflated every *_full number in every condition.
#    3. Per-epoch training loss AND a held-out conferral monitor, printed as it runs.
#    4. FACT_EPOCHS raised 6 -> 20, and the distinct-answer-token count is printed so
#       data starvation (the likely cause of recon > task) is visible immediately.
#
#  v3 CHANGES (after the run where best_step == 0 on all five seeds):
#    5. The task map is now a LOW-RANK RESIDUAL on the recon init: W_eff = FWr + A@B with
#       B initialized to zero, rank 32, weight-decayed. Step 0 is still exactly the recon
#       map, but rank 32 cannot memorize thousands of fact-specific directions, so the only
#       way for it to lower CE is a general read-out. FACT_RANK = 0 restores full rank.
#    6. FACT_LR 1e-3 -> 1e-4 (TRAIN conferral previously hit 0.94 by step 400), and the
#       held-out monitor now fires every 25 steps for the first 300.
#    7. Singleton answer classes are dropped from the CE loop ONLY. The ridge/recon map is
#       still fit on all of FT_TRAIN and the bin is untouched, so recon stays comparable.
#    8. CELL D defaults to the TriviaQA *train* split (~138k) at FACT_MAX = 40000; validation
#       holds only ~17.9k, so the old FACT_MAX = 8000 was starving the map.
#    9. If no seed beats the recon init, the run now says so in a banner and flags it in the
#       results JSON, instead of emitting recon numbers under a task_* label.
# =============================================================================
print("EXP27b — read the header, then run the dependency cells and restart the kernel")


EXP27b — read the header, then run the dependency cells and restart the kernel


In [3]:
# === CELL 1: Dependencies — run cells 0-3 once, in order, then RESTART KERNEL ===
# One resolved install so pip solves versions a SINGLE time, before any model is loaded:
#   * transformers is pinned to the version the model code targets (4.46.3)
#   * sae_lens (for EXP16 / EXP20 / EXP21) is installed HERE, not mid-run, so it can't retug
#     torch/transformers after the models are already sitting in memory.
# The cu128 torch fix in the NEXT cell runs AFTER this line, so on Blackwell / sm_120 pods it
# always wins over whatever torch sae_lens's resolver pulls. (If pip reports a hard transformers
# conflict from sae_lens, drop sae_lens from this line, `pip install sae_lens` on its own, then
# re-run this pinned line so transformers is restored to 4.46.3.)
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer sae_lens



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
!pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
!pip uninstall -y torchvision torchaudio

# typing_extensions repair. The cu128 --index-url in the previous cell REPLACES PyPI, so pip
# resolves every dependency from that index; it has no current typing_extensions, leaving a
# stale one while the upgraded torch needs >= 4.10 for `TypeIs`.
#   Why `sys.executable -m pip` and not `!pip`: on these images `!pip` is not always the pip
#   belonging to the kernel's interpreter, so the install can land somewhere not on sys.path.
#   Why --force-reinstall: a broken install can advertise a new version in its metadata (so pip
#   reports "already satisfied") while the .py file on disk is old.
import sys, subprocess, importlib.util
print("kernel python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall",
                "--no-cache-dir", "typing_extensions>=4.12"], check=False)
_s = importlib.util.find_spec("typing_extensions")
print("typing_extensions now at:", _s.origin if _s else "NOT FOUND")
print(">>> RESTART THE KERNEL before running the verify cell (the old module is cached in RAM).")


kernel python: /usr/bin/python



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


typing_extensions now at: /usr/local/lib/python3.11/dist-packages/typing_extensions.py
>>> RESTART THE KERNEL before running the verify cell (the old module is cached in RAM).


In [6]:
# verify the resolved stack BEFORE restarting (catch a bad resolve early, not 40 min into a run)
# NOTE: typing_extensions has no __version__ attribute, so do not print one. torch importing at
# all IS the check -- a stale typing_extensions makes `import torch` raise on `TypeIs`.
import torch
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
import transformers, sae_lens
print("transformers:", transformers.__version__, "(want 4.46.3)  | sae_lens:", sae_lens.__version__)


torch: 2.11.0+cu128 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)  | sae_lens: 6.5.3


In [7]:
# === CELL 2:imports, set_submodule shim, global config (VRAM/compute switches, primarily smoke test) ===
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
MODEL_2B = "google/gemma-2-2b"
MODEL_9B = "google/gemma-2-9b"

# Smoke test lever, (True for 5 min test eval False for full eval)
SMOKE_TEST = False            

# validated layers (Different for each model pair)
SINGLE_PAIR = (20, 34)
LAYER_PAIRS = [(18, 31), (20, 34), (22, 37), (24, 40)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 20, 24, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 300, 3000, 2000      # ~720 unsolvable muladd
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 150, 1319, [1.0]    # full GSM8K test set
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6                # 5 seeds for the task map
    BOOT_B = 10000

ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8
RESULTS = {}   # everything defensible gets concentrated here and printed at the end
print("SMOKE_TEST =", SMOKE_TEST)

SMOKE_TEST = False


In [ ]:
# === CELL 3: Hugging Face login (Gemma is gated) ===
from huggingface_hub import login
login("")

In [9]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [10]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook
 
def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m
 
def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2
 
# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
 
# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    m = _re.search(r"####\s*(-?[\d,.]+)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    m = _re.search(r"answer is\s*(-?[\d,.]+)", text, _re.IGNORECASE)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    nums = _re.findall(r"-?[\d,.]+", text)
    if nums:
        try: return float(nums[-1].rstrip(".").replace(",", ""))
        except ValueError: return None
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None
 
# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top
 
# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)
 
@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok
 
print("helpers defined")

helpers defined


In [11]:
# === CELL 6: load both models once; donor in bf16 by default (4-bit optional) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
 
# QUANTIZE_9B: set True only when the donor won't fit in bf16 (e.g. the original Gemma-2-9B on
# a small card). For 7-8B donors (Qwen-7B ~15GB, Llama-8B ~16GB) on a 20GB+ card, keep this
# False — bf16 is correct and avoids a serious failure mode: under 4-bit, this transformers/bnb
# build runs unquantized layers in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) ->
# NaN logits -> argmax collapses to token 0 ('!'). bf16 has the exponent range to avoid this.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
 
model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x2B layers,",
      model_9b.config.num_hidden_layers, "x9B layers |",
      "9B quantized" if QUANTIZE_9B else "9B bf16")
 
# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "9B produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl
 
# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

loaded: 26 x2B layers, 42 x9B layers | 9B bf16


In [12]:
# === RUN CONFIG — run this BEFORE CELL D and before EXP27b ===
# HARD ASSIGNMENTS, not globals().get() defaults. Your kernel from the previous run still holds
# FACT_MAX=8000, FACT_LR=1e-3, FACT_MAX_STEPS=2500, FACT_SEEDS=[0,1,2,3,4]; every globals().get()
# default further down would find those and silently keep the OLD value. This cell makes that
# impossible. (FACT_RANK / FACT_WD / FACT_MIN_CLASS_COUNT / FACT_SPLIT / FACT_MON_DENSE are new
# names, so they would pick up the new defaults anyway — they are listed here for one readout.)
FACT_SPLIT           = "train"   # TriviaQA rc.nocontext: ~138k rows; validation has only ~17.9k
FACT_MAX             = 40000     # candidates to encode (was 8000 -> 1.8 examples per class)
FACT_MIN_CLASS_COUNT = 2         # drop singleton answer classes from the CE LOOP ONLY
FACT_RANK            = 32        # low-rank residual on the recon init; 0 = old full-rank
FACT_WD              = 1e-2      # weight decay on A,B == anchoring toward the recon map
FACT_LR              = 1e-4      # was 1e-3: TRAIN conferral hit 0.94 by step 400
FACT_EPOCHS          = 4
FACT_MAX_STEPS       = 3000
FACT_MON_STEPS       = 100       # held-out check every N steps...
FACT_MON_DENSE       = 300       # ...but every 25 steps for the first 300
FACT_KEEP_BEST       = True
FACT_SEEDS           = [0, 1]    # DIAGNOSTIC. Go to [0,1,2,3,4] only once a seed beats recon.
RUN_CONFIG_APPLIED   = "v3"      # sentinel: EXP27b checks for this and refuses to run without it

# CELL D must be re-run: FACT_SPLIT/FACT_MAX changed, so facts.jsonl is rebuilt and the eval
# split (hence the bin) changes. The recon map is re-fit and re-measured in the SAME run, so the
# recon-vs-task comparison stays internally valid even though absolute numbers move.
for _k in ["FACT_SPLIT","FACT_MAX","FACT_MIN_CLASS_COUNT","FACT_RANK","FACT_WD","FACT_LR",
           "FACT_EPOCHS","FACT_MAX_STEPS","FACT_MON_STEPS","FACT_MON_DENSE","FACT_SEEDS"]:
    print(f"  {_k:22s} = {globals()[_k]}")


  FACT_SPLIT             = train
  FACT_MAX               = 40000
  FACT_MIN_CLASS_COUNT   = 2
  FACT_RANK              = 32
  FACT_WD                = 0.01
  FACT_LR                = 0.0001
  FACT_EPOCHS            = 4
  FACT_MAX_STEPS         = 3000
  FACT_MON_STEPS         = 100
  FACT_MON_DENSE         = 300
  FACT_SEEDS             = [0, 1]


In [13]:
# === CELL D: build a LONG-TAIL fact set for EXP27 (run before EXP27) ===
# The builtin capitals list produced an EMPTY bin: the 2B recipient solved 52/52. EXP27 needs
# facts the 9B DONOR gets right and the 2B RECIPIENT gets wrong, which means genuine long tail.
#
# SOURCES
#   "triviaqa"    naturalistic, human-written questions. Best answer to the paper's "our tasks
#                 are synthetic" limitation, and 2B models genuinely struggle on it.
#   "counterfact" ROME's subject-relation-object set. Templated, but very long-tailed.
#   "<path>"      your own JSONL, one {"prompt":..., "answer":...} per line (no build needed).
#
# HONESTY NOTE: I could not verify the HF config string or the CounterFact URL from this machine.
# Each loader PRINTS what it is attempting and raises loudly on failure rather than silently
# yielding a small set. If an identifier is wrong the traceback names it and you can fix it --
# check the dataset card on huggingface.co. Nothing below is silently best-effort.

FACT_BUILD = globals().get("FACT_BUILD", "triviaqa")   # "triviaqa" | "counterfact"
# TriviaQA rc.nocontext: validation is only ~17.9k records, so FACT_MAX above that SILENTLY
# caps. The train split has ~138k. We do our own dedup + train/eval split below, and the
# models never see labels from either, so "train" is just a larger fact pool -- not leakage.
FACT_SPLIT = globals().get("FACT_SPLIT", "train")      # "train" (~138k) | "validation" (~17.9k)
FACT_MAX   = globals().get("FACT_MAX", 40000)          # candidates to write; want >= ~5000
FACT_OUT   = globals().get("FACT_OUT", "facts.jsonl")
FACT_MAX_ANS_WORDS = globals().get("FACT_MAX_ANS_WORDS", 3)   # keep answers short

import json as _json, re as _re, os as _os

# exemplars are generic and deliberately NOT drawn from any of the sources below
_QA_FEWSHOT = ("Question: In which country is the city of Osaka?\nAnswer: Japan\n\n"
               "Question: Who wrote the play Hamlet?\nAnswer: Shakespeare\n\n"
               "Question: What is the chemical symbol for gold?\nAnswer: Au\n\n")

# CounterFact prompts are cloze stems ("The capital of X is"), so they get cloze exemplars
_CLOZE_FEWSHOT = ("Osaka is located in the country of Japan.\n"
                  "The chemical symbol for gold is Au.\n"
                  "Hamlet was written by Shakespeare.\n")

def _ok_answer(a):
    a = (a or "").strip()
    return bool(a) and len(a.split()) <= FACT_MAX_ANS_WORDS and a.isprintable()

rows = []

if FACT_BUILD == "triviaqa":
    print(f"CELL D: attempting  load_dataset('trivia_qa', 'rc.nocontext', split={FACT_SPLIT!r})")
    from datasets import load_dataset
    ds = load_dataset("trivia_qa", "rc.nocontext", split=FACT_SPLIT)
    print(f"  loaded {len(ds)} records | fields: {list(ds.features)[:8]}")
    for r in ds:
        q = (r.get("question") or "").strip()
        ans = r.get("answer") or {}
        a = (ans.get("value") or "").strip() if isinstance(ans, dict) else str(ans).strip()
        if not q or not _ok_answer(a): continue
        rows.append({"prompt": _QA_FEWSHOT + f"Question: {q}\nAnswer:", "answer": a})
        if len(rows) >= FACT_MAX: break

elif FACT_BUILD == "counterfact":
    URL = globals().get("COUNTERFACT_URL",
                        "https://rome.baulab.info/data/dsets/counterfact.json")
    print(f"CELL D: attempting download of CounterFact from {URL}")
    import urllib.request
    _local = "counterfact.json"
    if not _os.path.exists(_local):
        urllib.request.urlretrieve(URL, _local)
    recs = _json.load(open(_local))
    print(f"  loaded {len(recs)} records | first-record keys: {list(recs[0])}")
    for r in recs:
        rw = r.get("requested_rewrite") or {}
        tmpl, subj = rw.get("prompt"), rw.get("subject")
        tgt = (rw.get("target_true") or {}).get("str")
        if not (tmpl and subj and _ok_answer(tgt)): continue
        stem = tmpl.replace("{}", subj).strip()
        rows.append({"prompt": _CLOZE_FEWSHOT + stem, "answer": tgt.strip()})
        if len(rows) >= FACT_MAX: break

else:
    raise ValueError(f"FACT_BUILD must be 'triviaqa' or 'counterfact' "
                     f"(got {FACT_BUILD!r}); to use your own file set FACT_SOURCE directly")

if len(rows) < 500:
    raise RuntimeError(f"only {len(rows)} usable rows from {FACT_BUILD}. That will not give a "
                       f"usable bin. Check the loader above, or raise FACT_MAX / "
                       f"FACT_MAX_ANS_WORDS.")

# de-duplicate on prompt so the train/eval split cannot leak the same fact both ways
_seen, uniq = set(), []
for r in rows:
    if r["prompt"] in _seen: continue
    _seen.add(r["prompt"]); uniq.append(r)

with open(FACT_OUT, "w") as fh:
    for r in uniq:
        fh.write(_json.dumps(r) + "\n")

_alen = sum(len(r["answer"].split()) for r in uniq) / len(uniq)
print(f"\n  wrote {len(uniq)} unique facts -> {FACT_OUT}  (mean answer length {_alen:.2f} words)")
print("  sample:")
for r in uniq[:3]:
    print(f"    prompt tail ...{r['prompt'][-70:]!r}  -> {r['answer']!r}")
print(f"\n  NEXT: set  FACT_SOURCE = {FACT_OUT!r}  and run EXP27.")
print("  EXP27 prints donor/recipient accuracy and the bin size BEFORE it trains anything,")
print("  so if this source is still too easy you will see it in under a minute.")
FACT_SOURCE = FACT_OUT


CELL D: attempting  load_dataset('trivia_qa', 'rc.nocontext', split='train')


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

  loaded 138384 records | fields: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer']

  wrote 40000 unique facts -> facts.jsonl  (mean answer length 1.53 words)
  sample:
    prompt tail ...'ican-born Sinclair won the Nobel Prize for Literature in 1930?\nAnswer:'  -> 'Sinclair Lewis'
    prompt tail ...'swer: Au\n\nQuestion: Where in England was Dame Judi Dench born?\nAnswer:'  -> 'York'
    prompt tail ...'e did Billboard magazine first publish and American hit chart?\nAnswer:'  -> '30s'

  NEXT: set  FACT_SOURCE = 'facts.jsonl'  and run EXP27.
  EXP27 prints donor/recipient accuracy and the bin size BEFORE it trains anything,
  so if this source is still too easy you will see it in under a minute.


In [14]:
# === CELL E27b: EXP27 (FIXED) — NATURALISTIC FACTUAL RECALL ===
#
# WHAT WAS WRONG IN THE FIRST RUN, and what changed:
#
# (1) native_full = 0.146 on a bin where native_first is 0 BY CONSTRUCTION. Impossible if the
#     two metrics agreed, so they did not. The bin was built on EXACT FIRST TOKEN ID, but
#     full-answer was scored on the DECODED STRING. A model can reach the same string through a
#     different tokenization (' Paris' as one token vs ' Par' + 'is'), so items failed the token
#     check, entered the bin, and then passed the string check. Arithmetic never shows this
#     because digits tokenize canonically.
#     FIX: the bin is now defined on the DECODED FULL ANSWER for both models -- the same rule the
#     paper's cross-family arms already use. native_full is now 0 by construction; native_first
#     is nonzero and reported, exactly as in the cross-family setting.
#
# (2) `startswith` matching was far too loose: gold "Au" matched "Australia", gold "US" matched
#     "used to be a colony". TriviaQA is full of 2-4 character answers, so this inflated EVERY
#     *_full number in every condition.
#     FIX: strict match -- normalized equality, or gold followed by a word boundary.
#
# (3) recon (0.251) BEAT task (0.146), inverted from every same-family result. Most likely the
#     task map is data-starved: arithmetic has 9 answer classes over 3000 items, TriviaQA has
#     thousands over 4800. The cell now PRINTS the distinct-answer-token count, prints per-epoch
#     loss, and periodically prints held-out conferral so underfitting is visible while it runs.
#
# REUSES from the setup cells above: states_and_top, left_pad, fit_ridge, patch_vec_batch,
# _graft, wilson_bools, across_seed_ci, fmt, L2_SINGLE, L9_SINGLE, ARITH_BATCH, DEVICE, RESULTS,
# tokenizer, model_2b, model_9b.

RUN_FACTS       = globals().get("RUN_FACTS", True)
FACT_SOURCE     = globals().get("FACT_SOURCE", "facts.jsonl")
FACT_TRAIN_FRAC = globals().get("FACT_TRAIN_FRAC", 0.6)
FACT_SEEDS      = globals().get("FACT_SEEDS", [0, 1, 2, 3, 4])
FACT_EPOCHS     = globals().get("FACT_EPOCHS", 4)      # held-out conferral peaked at epoch 2
#                 more epochs OVERFIT: train CE -> 0.15 while held-out fell. Do not raise this
#                 to fix underperformance; fix examples-per-class in CELL D instead.
# The v3 fix. The previous run ended with best_step == 0 on all five seeds: full-rank Adam at
# lr=1e-3 drove TRAIN conferral 0.33 -> 0.99 inside one epoch while held-out sat at 0.13-0.19,
# so FACT_KEEP_BEST restored the recon init and every "task" number in the JSON was the recon
# map. Three changes, all aimed at "can it learn a GENERAL read-out rather than memorize":
#   FACT_RANK  low-rank residual on the recon init: W_eff = FWr + A@B, B initialized to ZERO so
#              step 0 is still exactly the recon map. Rank 32 cannot store thousands of
#              fact-specific directions, so the only way to lower CE is a general correction.
#              Set FACT_RANK = 0 to recover the old full-rank behaviour for an A/B.
#   FACT_LR    1e-3 -> 1e-4. Train conferral hit 0.94 by step 400; that is far too fast.
#   FACT_WD    weight decay on A,B only. Decaying them toward 0 IS anchoring to the recon map,
#              which is the right prior here. Never applied to the bias.
FACT_RANK       = globals().get("FACT_RANK", 32)       # 0 = full-rank (old behaviour)
FACT_WD         = globals().get("FACT_WD", 1e-2)       # weight decay on the low-rank residual
FACT_LR         = globals().get("FACT_LR", 1e-4)
# Train-only class filter. Singleton answer classes cannot teach a generalizable read-out --
# one example is pure memorization fuel. This drops them from the CE loop ONLY. The ridge/recon
# map is still fit on ALL of FT_TRAIN (line ~146) and FT_EVAL/the bin are untouched, so the
# recon baseline stays comparable to the previous run. Set 1 to disable.
FACT_MIN_CLASS_COUNT = globals().get("FACT_MIN_CLASS_COUNT", 2)
FACT_EVAL_EVERY = globals().get("FACT_EVAL_EVERY", 2)  # print held-out conferral every N epochs
FACT_MON_N      = globals().get("FACT_MON_N", 160)     # items used for that quick monitor
# The task map is warm-started AT the recon map (W=FWr, b=Fmu2), so it begins at recon's score
# and can only be judged by whether training moves it UP. Epoch-level monitoring was far too
# coarse: one epoch here is 2250 Adam steps (the arithmetic map takes 1128 steps IN TOTAL), so
# the first measurement already came long after any optimum. Monitor by STEP instead.
FACT_MON_STEPS  = globals().get("FACT_MON_STEPS", 100)   # held-out check every N optimizer steps
FACT_MON_DENSE  = globals().get("FACT_MON_DENSE", 300)   # ...every 25 steps for the first N
FACT_MAX_STEPS  = globals().get("FACT_MAX_STEPS", 3000)  # hard cap on total steps per seed
FACT_KEEP_BEST  = globals().get("FACT_KEEP_BEST", True)  # keep the best-held-out checkpoint
MAX_NEW_FACT    = globals().get("MAX_NEW_FACT", 12)
FACT_MAX_EVAL   = globals().get("FACT_MAX_EVAL", 3000) # cap: bin build needs generation x2 models

# Guard: every knob above is a globals().get() default, so a value left in the kernel by an
# earlier run silently wins. That has already happened three times in this experiment. The RUN
# CONFIG cell hard-assigns them and sets this sentinel; without it, stop rather than burn hours
# on stale settings. Then echo what is ACTUALLY in effect -- one readout, no inference.
assert globals().get("RUN_CONFIG_APPLIED") == "v3", (
    "RUN CONFIG cell was not run (it sits just above CELL D). Run it first: without it this cell "
    "would reuse FACT_MAX/FACT_LR/FACT_MAX_STEPS/FACT_SEEDS left over from your previous run.")
print("EXP27b running with:  seeds =", FACT_SEEDS, " rank =", FACT_RANK, " lr =", FACT_LR,
      " wd =", FACT_WD, " epochs =", FACT_EPOCHS, " max_steps =", FACT_MAX_STEPS,
      " min_class_count =", FACT_MIN_CLASS_COUNT)

FACT_OK = False
if RUN_FACTS:
    import json as _json

    def _norm(t):
        return "".join(c for c in t.lower() if c.isascii() and (c.isalnum() or c == " ")).strip()

    def _match(gen_text, gold):
        """Strict: normalized equality, or gold followed by a word boundary.
        Rejects the old failure mode where gold 'Au' matched 'Australia'."""
        g, p = _norm(gold), _norm(gen_text.split("\n")[0])
        return bool(g) and (p == g or p.startswith(g + " "))

    def _first_answer_token(tok, prompt, answer):
        p = tok(prompt).input_ids
        f = tok(prompt + " " + answer).input_ids
        if f[:len(p)] != p or len(f) <= len(p):
            return None, None
        j = len(p)
        while j < len(f) and tok.decode([f[j]]).strip() == "":
            j += 1
        return (torch.tensor(f[:j]), f[j]) if j < len(f) else (None, None)

    _raw = []
    with open(FACT_SOURCE) as fh:
        for line in fh:
            line = line.strip()
            if line: _raw.append(_json.loads(line))
    FACTS = []
    for r in _raw:
        ids, tid = _first_answer_token(tokenizer, r["prompt"], r["answer"])
        if tid is not None:
            FACTS.append(dict(prompt=r["prompt"], answer=r["answer"], ids=ids, tok=tid))
    print(f"E27b: {len(_raw)} candidates -> {len(FACTS)} encoded")

    _fi = list(range(len(FACTS))); random.Random(0).shuffle(_fi)
    _cut = max(1, int(len(_fi) * FACT_TRAIN_FRAC))
    FT_TRAIN = [FACTS[i] for i in _fi[:_cut]]
    FT_EVAL  = [FACTS[i] for i in _fi[_cut:]][:FACT_MAX_EVAL]
    _ntok = len({p["tok"] for p in FT_TRAIN})
    print(f"      train={len(FT_TRAIN)} eval={len(FT_EVAL)}")
    print(f"      DISTINCT ANSWER TOKENS IN TRAIN = {_ntok}  "
          f"({len(FT_TRAIN)/max(1,_ntok):.1f} examples per class)")
    if _ntok > len(FT_TRAIN) / 20:
        print("      >>> WARNING: very few examples per answer class. A cross-entropy task map")
        print("      >>> may be data-starved here; watch the per-epoch loss and monitor below.")
    FACT_OK = len(FT_EVAL) >= 50

if FACT_OK:
    @torch.inference_mode()
    def _generate(model, items, vec_fn=None, layer=None):
        """Free-generate for `items`. If vec_fn is given, graft vec_fn(batch_slice) first."""
        outs, h = [], None
        if vec_fn is not None:
            h = model.model.layers[layer].register_forward_hook(patch_vec_batch)
        try:
            for i in range(0, len(items), ARITH_BATCH):
                chunk = items[i:i + ARITH_BATCH]
                ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
                if vec_fn is not None:
                    v = vec_fn(i, len(chunk))
                    assert v.shape[0] == len(chunk), f"graft {v.shape[0]} != {len(chunk)}"
                    _graft["vec"] = v
                gen = model.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                     max_new_tokens=MAX_NEW_FACT, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
                outs += tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
        finally:
            if h is not None: h.remove()
            _graft["vec"] = None
        return outs

    # ---- BIN: decoded full answer, both models. native_full is now 0 by construction. ----
    print("      building bin by GENERATION (both models) — this is the slow part ...")
    _d_txt = _generate(model_9b, FT_EVAL)
    _r_txt = _generate(model_2b, FT_EVAL)
    F_donor_ok = [_match(_d_txt[i], FT_EVAL[i]["answer"]) for i in range(len(FT_EVAL))]
    F_recip_ok = [_match(_r_txt[i], FT_EVAL[i]["answer"]) for i in range(len(FT_EVAL))]
    F_UNSOLV = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and not F_recip_ok[i]]
    F_SOLV   = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and F_recip_ok[i]]
    print(f"      donor {sum(F_donor_ok)}/{len(FT_EVAL)} | recipient {sum(F_recip_ok)}/{len(FT_EVAL)}")
    print(f"      >>> UNSOLVABLE BIN n = {len(F_UNSOLV)}")

    # native FIRST-token on this bin is NOT zero (bin is full-answer defined) — report it
    _, F2_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_EVAL])
    F_nat_first = [F2_top[j] == FT_EVAL[j]["tok"] for j in F_UNSOLV]
    F_nat_full  = [_match(_r_txt[j], FT_EVAL[j]["answer"]) for j in F_UNSOLV]   # 0 by construction
    print(f"      native on bin: first={sum(F_nat_first)/max(1,len(F_UNSOLV)):.3f} "
          f"full={sum(F_nat_full)/max(1,len(F_UNSOLV)):.3f} (full must be 0.000)")
    FACT_OK = len(F_UNSOLV) >= 50
    if not FACT_OK: print("EXP27b STOPPED: bin < 50.")

if FACT_OK:
    FX9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in FT_TRAIN]); FX9t = FX9t[L9_SINGLE]
    FX2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_TRAIN]); FX2t = FX2t[L2_SINGLE]
    FX9e, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in FT_EVAL]);  FX9e = FX9e[L9_SINGLE]
    FX2e, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_EVAL]);  FX2e = FX2e[L2_SINGLE]
    Fmu9, Fmu2, FWr = fit_ridge(FX9t, FX2t)
    Fmu9d, Fmu2d, FWr_d = Fmu9.to(DEVICE), Fmu2.to(DEVICE), FWr.to(DEVICE)

    @torch.inference_mode()
    def F_first(vec_fn, idxs):
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i + ARITH_BATCH]
                v = vec_fn(sub); assert v.shape[0] == len(sub)
                _graft["vec"] = v
                ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == FT_EVAL[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return out

    def F_full(vec_fn, idxs):
        items = [FT_EVAL[j] for j in idxs]
        def _vf(i, n): return vec_fn(idxs[i:i + n])
        txt = _generate(model_2b, items, vec_fn=_vf, layer=L2_SINGLE)
        return [_match(txt[k], items[k]["answer"]) for k in range(len(items))]

    MON = F_UNSOLV[:FACT_MON_N]

    # ---- task-map training subset (CE loop only; ridge and bin untouched) ----
    from collections import Counter as _Counter
    _cc = _Counter(p["tok"] for p in FT_TRAIN)
    FT_TASK_IDX = [i for i, p in enumerate(FT_TRAIN) if _cc[p["tok"]] >= FACT_MIN_CLASS_COUNT]
    _ncls_task = len({FT_TRAIN[i]["tok"] for i in FT_TASK_IDX})
    _epc_task = len(FT_TASK_IDX) / max(1, _ncls_task)
    print(f"      CE training subset: {len(FT_TASK_IDX)}/{len(FT_TRAIN)} items over "
          f"{_ncls_task} classes ({_epc_task:.1f} per class), min class count "
          f"= {FACT_MIN_CLASS_COUNT}  [was {len(FT_TRAIN)/max(1,_ntok):.1f} per class]")
    print(f"      ridge/recon map still fit on ALL {len(FT_TRAIN)} items; bin unchanged")
    assert len(FT_TASK_IDX) >= 200, (
        f"only {len(FT_TASK_IDX)} items survive FACT_MIN_CLASS_COUNT="
        f"{FACT_MIN_CLASS_COUNT}; lower it or raise FACT_MAX in CELL D")

    # Memorization probe: the same measurement on TRAINING items the map has already seen.
    # If train conferral climbs while held-out falls, the map is memorizing fact-specific
    # directions rather than learning a general read-out. That is the decisive test for
    # "what is it converging to".
    _TRMON = FT_TASK_IDX[:FACT_MON_N]   # probe items the CE loop actually trains on
    @torch.inference_mode()
    def F_first_train(W_, b_):
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(_TRMON), ARITH_BATCH):
                sub = _TRMON[i:i + ARITH_BATCH]
                _graft["vec"] = (FX9t[sub].to(DEVICE) - Fmu9d) @ W_ + b_
                ids, m = left_pad([FT_TRAIN[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == FT_TRAIN[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return sum(out) / max(1, len(out))
    F_task_maps  = []
    F_best_step  = []          # per seed; 0 means training never beat the recon init
    _d9, _d2 = FWr.shape
    for sd in FACT_SEEDS:
        torch.manual_seed(sd)
        b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
        if FACT_RANK and FACT_RANK > 0:
            # LoRA-style init: A ~ small random, B = 0  =>  W_eff == FWr exactly at step 0, so the
            # first monitor row is still the recon map and "did training help" stays readable.
            A = (torch.randn(_d9, FACT_RANK, device=DEVICE) / math.sqrt(_d9)).requires_grad_(True)
            B = torch.zeros(FACT_RANK, _d2, device=DEVICE).requires_grad_(True)
            groups = [{"params": [A, B], "weight_decay": FACT_WD},
                      {"params": [b],    "weight_decay": 0.0}]
            def _W_eff(): return FWr_d + A @ B
            def _vec(x):
                # never materialize the 3584x2304 product per step: [16,d9]->[16,r]->[16,d2]
                xc = x - Fmu9d
                return xc @ FWr_d + (xc @ A) @ B + b
        else:
            Wf = FWr.clone().to(DEVICE).requires_grad_(True)
            groups = [{"params": [Wf, b], "weight_decay": 0.0}]
            def _W_eff(): return Wf
            def _vec(x): return (x - Fmu9d) @ Wf + b
        opt = torch.optim.Adam(groups, lr=FACT_LR)
        model_2b.requires_grad_(False)
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            idx = list(FT_TASK_IDX)
            gstep, stop = 0, False
            best_W, best_b, best_step = _W_eff().detach().clone(), b.detach().clone(), 0
            # step 0 = the recon map itself, so the very first row is the baseline to beat
            _m0 = F_first(lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ best_W + best_b, MON)
            best_acc = sum(_m0) / len(_m0)
            print(f"      seed {sd} step     0 (= recon init)  TRAIN = {F_first_train(best_W, best_b):.3f}"
                  f"   held-out = {best_acc:.3f}   |dW|/|W| = 0.000", flush=True)
            for ep in range(FACT_EPOCHS):
                if stop: break
                random.Random(100 * sd + ep).shuffle(idx)
                for st in range(0, len(idx), ARITH_BATCH):
                    sub = idx[st:st + ARITH_BATCH]
                    _graft["vec"] = _vec(FX9t[sub].to(DEVICE))
                    ids, m = left_pad([FT_TRAIN[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([FT_TRAIN[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(logits, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    gstep += 1
                    # dense early monitoring: the previous run's first measurement landed at step
                    # 100, by which point TRAIN conferral was already 0.39 and climbing fast.
                    if (gstep % FACT_MON_STEPS == 0) or (gstep <= FACT_MON_DENSE and gstep % 25 == 0):
                        _Wd, _bd = _W_eff().detach().clone(), b.detach().clone()
                        _mon = F_first(lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ _Wd + _bd, MON)
                        _acc = sum(_mon) / len(_mon)
                        flag = ""
                        if _acc > best_acc:
                            best_acc, best_W, best_b, best_step = _acc, _Wd, _bd, gstep
                            flag = "  <- best"
                        _tr = F_first_train(_Wd, _bd)
                        _dw = float((_Wd - FWr_d).norm() / FWr_d.norm())
                        print(f"      seed {sd} step {gstep:5d} (ep {ep})  CE = {float(loss.item()):.4f}"
                              f"   TRAIN = {_tr:.3f}   held-out = {_acc:.3f}"
                              f"   |dW|/|W| = {_dw:.3f}{flag}", flush=True)
                    if gstep >= FACT_MAX_STEPS:
                        stop = True; break
            print(f"      seed {sd} BEST held-out = {best_acc:.3f} at step {best_step}"
                  f"{'  (never beat the recon init -> this seed IS the recon map)' if best_step == 0 else ''}",
                  flush=True)
            if FACT_KEEP_BEST:
                W_out, b_out = best_W, best_b
            else:
                W_out, b_out = _W_eff().detach().clone(), b.detach().clone()
        finally:
            h.remove(); _graft["vec"] = None
            model_2b.requires_grad_(True)
        F_best_step.append(best_step)
        F_task_maps.append((W_out.detach().clone(), b_out.detach().clone()))

    # Guard against the previous run's reporting hazard: when best_step == 0 the kept checkpoint
    # is literally the recon init, so every "task_*" number below is the recon map wearing a task
    # label (that is why the last run printed five identical per-seed values and an across-seed
    # interval of [0.217, 0.217]). Make it impossible to read past.
    F_TASK_IS_RECON = [bs == 0 for bs in F_best_step]
    if all(F_TASK_IS_RECON):
        print("\n      " + "!" * 68)
        print("      !! NO SEED BEAT THE RECON INIT. Every task_* number below is the recon")
        print("      !! map, not a trained task map. Do not report them as a task map.")
        print("      " + "!" * 68 + "\n", flush=True)

    F_recon = lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d
    def F_task(i):
        W, b = F_task_maps[i]
        return lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ W + b
    F_self  = lambda sub: FX2e[sub].to(DEVICE).float()
    def _fpartner(idxs, seed):
        rng = random.Random(seed); pool = list(idxs); perm = pool[:]
        for _ in range(200):
            rng.shuffle(perm)
            if all(FT_EVAL[perm[k]]["tok"] != FT_EVAL[pool[k]]["tok"] for k in range(len(pool))):
                return dict(zip(pool, perm)), 0
        return dict(zip(pool, perm)), sum(
            1 for k in range(len(pool)) if FT_EVAL[perm[k]]["tok"] == FT_EVAL[pool[k]]["tok"])
    F_PART, _fcol = _fpartner(F_UNSOLV, 7)
    def F_shuf(sub):
        W0, b0 = F_task_maps[0]
        return (FX9e[[F_PART[j] for j in sub]].to(DEVICE) - Fmu9d) @ W0 + b0

    # RECON's own shuffled control. Without this we cannot distinguish two explanations for
    # recon lifting first-token 0.061 -> 0.289:
    #   (a) it transfers this problem's donor content  -> shuffled recon should COLLAPSE
    #   (b) writing any well-formed, on-manifold vector papers over the recipient's errors
    #       -> shuffled recon should lift it just as much
    # (b) is the "it is only fixing minor glitches" hypothesis. This is the test that settles it.
    def F_recon_shuf(sub):
        return (FX9e[[F_PART[j] for j in sub]].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d

    _seed_first = [sum(F_first(F_task(i), F_UNSOLV)) / len(F_UNSOLV) for i in range(len(F_task_maps))]
    RESULTS["factual_recall_v2"] = {
        "_what": ("naturalistic factual recall. BIN IS DEFINED ON THE DECODED FULL ANSWER for "
                  "both models (same rule as the paper's cross-family arms), so native full-answer "
                  "is 0 by construction and native FIRST-token is nonzero."),
        "_fixes": ["bin now full-answer defined (was first-token, which disagreed with the "
                   "full-answer metric via tokenization non-determinism)",
                   "strict answer matching (was startswith: 'Au' matched 'Australia')",
                   "per-epoch loss + held-out conferral monitor",
                   f"v3: low-rank residual on the recon init (rank {FACT_RANK}), lr {FACT_LR}, "
                   f"wd {FACT_WD}, singleton answer classes dropped from the CE loop only",
                   f"epochs {FACT_EPOCHS}"],
        "source": FACT_SOURCE, "n_encoded": len(FACTS),
        "n_train": len(FT_TRAIN), "n_eval": len(FT_EVAL),
        "distinct_answer_tokens_in_train": _ntok,
        "examples_per_answer_class": round(len(FT_TRAIN) / max(1, _ntok), 2),
        "task_map_cfg": {"rank": FACT_RANK, "lr": FACT_LR, "weight_decay": FACT_WD,
                         "min_class_count": FACT_MIN_CLASS_COUNT, "epochs": FACT_EPOCHS,
                         "max_steps": FACT_MAX_STEPS, "keep_best": FACT_KEEP_BEST},
        "ce_train_items": len(FT_TASK_IDX), "ce_train_classes": _ncls_task,
        "ce_examples_per_class": round(_epc_task, 2),
        "donor_full_acc": round(sum(F_donor_ok) / len(FT_EVAL), 4),
        "recipient_full_acc": round(sum(F_recip_ok) / len(FT_EVAL), 4),
        "n_unsolv": len(F_UNSOLV), "n_solv": len(F_SOLV),
        "native_first":    fmt(wilson_bools(F_nat_first)),
        "native_full":     fmt(wilson_bools(F_nat_full)),
        "selfgraft_first": fmt(wilson_bools(F_first(F_self,  F_UNSOLV))),
        "shuffle_first":   fmt(wilson_bools(F_first(F_shuf,  F_UNSOLV))),
        "_shuffle_note": ("shuffle_first uses F_task_maps[0]; when that seed is the recon init "
                          "this is the SAME computation as recon_SHUFFLED_first, so agreement "
                          "between them is a consistency check, not two independent controls"),
        "recon_first":     fmt(wilson_bools(F_first(F_recon, F_UNSOLV))),
        "recon_SHUFFLED_first": fmt(wilson_bools(F_first(F_recon_shuf, F_UNSOLV))),
        "_recon_shuffle_note": ("if this is near native (0.06) the recon lift is problem-specific "
                                "conferral; if it is near recon_first the lift is generic, i.e. "
                                "writing any on-manifold vector repairs the recipient"),
        "task_best_step_per_seed": F_best_step,
        "task_is_recon_init_per_seed": F_TASK_IS_RECON,
        "_task_warning": ("ALL SEEDS ARE THE RECON INIT -- every task_* field below is the recon "
                          "map, not a trained task map; do not report them as one"
                          if all(F_TASK_IS_RECON) else
                          "some seeds improved on the recon init; see task_best_step_per_seed"),
        "task_first_seed0": fmt(wilson_bools(F_first(F_task(0), F_UNSOLV))),
        "task_first_per_seed": [round(v, 4) for v in _seed_first],
        "task_first_acrossseed": fmt(across_seed_ci(_seed_first)) if len(_seed_first) > 1 else "n/a",
        "recon_full":       fmt(wilson_bools(F_full(F_recon, F_UNSOLV))),
        "task_full_seed0":  fmt(wilson_bools(F_full(F_task(0), F_UNSOLV))),
        "shuffled_pairing": {"n_pairs": len(F_PART), "unavoidable_same_token": _fcol},
    }
    print("EXP27b:", json.dumps(RESULTS["factual_recall_v2"], indent=2))
elif RUN_FACTS:
    print("EXP27b did not run to completion — see the stop message above.")


EXP27b running with:  seeds = [0, 1]  rank = 32  lr = 0.0001  wd = 0.01  epochs = 4  max_steps = 3000  min_class_count = 2
E27b: 40000 candidates -> 40000 encoded
      train=24000 eval=3000
      DISTINCT ANSWER TOKENS IN TRAIN = 7252  (3.3 examples per class)
      >>> WARNING: very few examples per answer class. A cross-entropy task map
      >>> may be data-starved here; watch the per-epoch loss and monitor below.
      building bin by GENERATION (both models) — this is the slow part ...
      donor 1976/3000 | recipient 1505/3000
      >>> UNSOLVABLE BIN n = 570
      native on bin: first=0.065 full=0.000 (full must be 0.000)
      CE training subset: 20348/24000 items over 3600 classes (5.7 per class), min class count = 2  [was 3.3 per class]
      ridge/recon map still fit on ALL 24000 items; bin unchanged
      seed 0 step     0 (= recon init)  TRAIN = 0.556   held-out = 0.287   |dW|/|W| = 0.000
      seed 0 step    25 (ep 0)  CE = 1.9880   TRAIN = 0.556   held-out = 0.287   |d

In [15]:
import json, datetime
with open("exp27b_results.json","w") as fh:
    json.dump({"factual_recall_v2": RESULTS["factual_recall_v2"]}, fh, indent=2)
print("wrote exp27b_results.json at", datetime.datetime.now().isoformat(timespec="seconds"))


wrote exp27b_results.json at 2026-09-10T08:15:38
